In [10]:
# Install Databricks SQL Connector
!pip install databricks-sql-connector pandas -q

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 4, in <module>
    from pip._internal.cli.main import main
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 11, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main_parser.py", line 9, in <module>
    from pip._internal.build_env import get_runnable_pip
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/build_env.py", line 19, in <module>
    from pip._internal.cli.spi

In [11]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
    print("✅ Drive mounted successfully")
else:
    print("✅ Drive already mounted, skipping...")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
#configuration notebook

import os
os.chdir("/content/drive/My Drive/Colab Notebooks")
%run oo_config.ipynb

In [13]:
# ─── Databricks Connection Config ───────────────────────────────────────────
SERVER_HOSTNAME = G_SERVER_HOSTNAME
HTTP_PATH       = G_HTTP_PATH   # from Connection Details tab
ACCESS_TOKEN    = G_ACCESS_TOKEN         # replace with your PAT

# ─── Catalog / Schema / Table Target ────────────────────────────────────────
CATALOG   = "workspace"            # change to your catalog name (Unity Catalog)
SCHEMA    = "default"         # change to your schema/database
TABLE     = "test_colab_insert"
FULL_TABLE = f"{CATALOG}.{SCHEMA}.{TABLE}"

In [14]:
import pandas as pd

# Simulate a Spark DataFrame using Pandas (Colab has no Spark cluster)
data = {
    "id":        [1, 2, 3, 4, 5],
    "name":      ["Alice", "Bob", "Charlie", "Diana", "Eve"],
    "score":     [92.5, 87.0, 78.3, 95.1, 88.9],
    "is_active": [True, False, True, True, False]
}

pdf = pd.DataFrame(data)
print("Sample DataFrame:")
print(pdf)


Sample DataFrame:
   id     name  score  is_active
0   1    Alice   92.5       True
1   2      Bob   87.0      False
2   3  Charlie   78.3       True
3   4    Diana   95.1       True
4   5      Eve   88.9      False


In [15]:
from databricks import sql

def run_query(cursor, query):
    """Helper to execute and optionally fetch results."""
    cursor.execute(query)
    try:
        return cursor.fetchall()
    except Exception:
        return None

with sql.connect(
    server_hostname = SERVER_HOSTNAME,
    http_path       = HTTP_PATH,
    access_token    = ACCESS_TOKEN
) as conn:
    with conn.cursor() as cursor:

        # ── Step 1: Set catalog and schema context ───────────────────────────
        cursor.execute(f"USE CATALOG {CATALOG}")
        cursor.execute(f"USE SCHEMA {SCHEMA}")

        # ── Step 2: Create table if not exists ───────────────────────────────
        create_sql = f"""
        CREATE TABLE IF NOT EXISTS {FULL_TABLE} (
            id        INT,
            name      STRING,
            score     DOUBLE,
            is_active BOOLEAN
        )
        """
        cursor.execute(create_sql)
        print(f"✅ Table '{FULL_TABLE}' created/verified.")

        # ── Step 3: Insert rows from Pandas DataFrame ─────────────────────────
        #for _, row in pdf.iterrows():
        #    insert_sql = f"""
        #    INSERT INTO {FULL_TABLE} (id, name, score, is_active)
        #    VALUES ({row['id']}, '{row['name']}', {row['score']}, {str(row['is_active']).upper()})
        #    """
        #    cursor.execute(insert_sql)

        rows = [tuple(r) for r in pdf.itertuples(index=False)]

        cursor.executemany(
            f"INSERT INTO {FULL_TABLE} VALUES (?, ?, ?, ?)",
            rows
        )

        print(f"✅ Inserted {len(pdf)} rows into '{FULL_TABLE}'.")

        # ── Step 4: Verify by reading back ───────────────────────────────────
        results = run_query(cursor, f"SELECT * FROM {FULL_TABLE}")
        print(f"\n📦 Data in '{FULL_TABLE}':")
        for r in results:
            print(r)

✅ Table 'workspace.default.test_colab_insert' created/verified.
✅ Inserted 5 rows into 'workspace.default.test_colab_insert'.

📦 Data in 'workspace.default.test_colab_insert':
Row(id=3, name='Charlie', score=78.30000305175781, is_active=True)
Row(id=3, name='Charlie', score=78.30000305175781, is_active=True)
Row(id=3, name='Charlie', score=78.30000305175781, is_active=True)
Row(id=3, name='Charlie', score=78.30000305175781, is_active=True)
Row(id=3, name='Charlie', score=78.30000305175781, is_active=True)
Row(id=3, name='Charlie', score=78.3, is_active=True)
Row(id=4, name='Diana', score=95.0999984741211, is_active=True)
Row(id=4, name='Diana', score=95.0999984741211, is_active=True)
Row(id=4, name='Diana', score=95.1, is_active=True)
Row(id=4, name='Diana', score=95.0999984741211, is_active=True)
Row(id=4, name='Diana', score=95.0999984741211, is_active=True)
Row(id=4, name='Diana', score=95.0999984741211, is_active=True)
Row(id=1, name='Alice', score=92.5, is_active=True)
Row(id=1, n

In [16]:
# More efficient for larger DataFrames — uses parameterized batch insert
%%script echo "Skipping..."
'''
with sql.connect(
    server_hostname = SERVER_HOSTNAME,
    http_path       = HTTP_PATH,
    access_token    = ACCESS_TOKEN
) as conn:
    with conn.cursor() as cursor:

        rows = [tuple(r) for r in pdf.itertuples(index=False)]

        cursor.executemany(
            f"INSERT INTO {FULL_TABLE} VALUES (?, ?, ?, ?)",
            rows
        )
        print(f"✅ Batch inserted {len(rows)} rows via executemany.")
'''

Skipping...
